In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.describe()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
count,4631.000000,5000.000000,5000.000000,4965.000000,5000.000000
mean,54533.212049,2.089400,4.472600,0.494767,0.576600
std,21245.661710,1.188987,2.435906,0.199431,0.494147
min,12000.000000,0.000000,0.000000,0.010000,0.000000
25%,40458.500000,1.000000,3.000000,0.360000,0.000000
50%,55051.000000,2.000000,4.000000,0.490000,1.000000
75%,71252.500000,3.000000,6.000000,0.630000,1.000000
max,109886.000000,6.000000,13.000000,0.990000,1.000000


In [4]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [5]:
df.lead_source = df.lead_source.fillna('NA')
df.industry = df.industry.fillna('NA')
df.employment_status = df.employment_status.fillna('NA')
df.location = df.location.fillna('NA')

df.annual_income = df.annual_income.fillna(0.0)
df.lead_score = df.lead_score.fillna(0.0)

### Question 1. (technology)

In [6]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

### Question 2.(interaction_count - lead_score)

In [7]:
correlation_matrix = df.corr(numeric_only=True)
print(correlation_matrix)

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   
converted                      0.189652                  0.395038   

                          interaction_count  lead_score  converted  
annual_income                      0.122842    0.229496   0.189652  
number_of_courses_viewed           0.721609    0.757204   0.395038  
interaction_count                  1.000000    0.915746   0.448624  
lead_score                         0.915746    1.000000   0.483246  
converted                          0.448624    0.483246   1.000000  


In [8]:
correlation = df['interaction_count'].corr(df['lead_score'])

print(f"Коэффициент корреляции: {correlation}")

Коэффициент корреляции: 0.9157457980418677


In [9]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split( df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values



### Question 3. (lead_source)

In [10]:
from sklearn.metrics import mutual_info_score

def mutual_info_converted_score(series):
    return mutual_info_score(series, df_train.converted)

categorical = ["lead_source", "industry", "employment_status", "location"]
numerical = ["annual_income","number_of_courses_viewed","interaction_count", "lead_score"]
			

mi = df_train[categorical].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

In [11]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

### Question 4. (0.65)

In [12]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)


In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
accuracy = (y_val == converted_decision).mean()
print(accuracy)


0.645


### Question 6 (0.001)

In [14]:
for creg in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=creg, max_iter=1000, random_state=42)
    # solver='lbfgs' is the default solver in newer version of sklearn
    # for older versions, you need to specify it explicitly
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)
    print((y_val == converted_decision).mean())
    

0.598
0.598
0.613
0.645


### Question 5 (number_of_courses_viewed)

In [15]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

features = categorical + numerical
n=len(features)

for i in range(n):
    features_new = features[0:i] + features[i+1:]

    dv = DictVectorizer(sparse=False)
    
    train_dict = df_train[features_new].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    
    val_dict = df_val[features_new].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    # solver='lbfgs' is the default solver in newer version of sklearn
    # for older versions, you need to specify it explicitly
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)
    accuracy_new = (y_val == converted_decision).mean()
    print(f"excluded feature {features[i]}: {accuracy_new}, difference: {abs(accuracy - accuracy_new)}")

excluded feature lead_source: 0.642, difference: 0.0030000000000000027
excluded feature industry: 0.645, difference: 0.0
excluded feature employment_status: 0.645, difference: 0.0
excluded feature location: 0.645, difference: 0.0
excluded feature annual_income: 0.724, difference: 0.07899999999999996
excluded feature number_of_courses_viewed: 0.643, difference: 0.0020000000000000018
excluded feature interaction_count: 0.601, difference: 0.04400000000000004
excluded feature lead_score: 0.644, difference: 0.0010000000000000009


### Converted rate and risk ratio

In [17]:
from IPython.display import display

global_converted = df_full_train.converted.mean()
print(global_converted)

for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).converted.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_converted
    df_group['risk'] = df_group['mean'] / global_converted
    display(df_group)
    print()
    print()

0.57075
lead_source


,mean,count,diff,risk
lead_source,,,,
NA,0.610169,118,0.039419,1.069066
events,0.411290,496,-0.159460,0.720614
organic_search,0.677579,1008,0.106829,1.187174
paid_ads,0.427957,930,-0.142793,0.749815
referral,0.763844,614,0.193094,1.338316
social_media,0.547962,834,-0.022788,0.960073




industry


,mean,count,diff,risk
industry,,,,
NA,0.587629,194,0.016879,1.029573
education,0.500000,518,-0.070750,0.876040
finance,0.613402,582,0.042652,1.074730
healthcare,0.559398,665,-0.011352,0.980111
manufacturing,0.548969,388,-0.021781,0.961838
retail,0.553741,735,-0.017009,0.970200
technology,0.611111,918,0.040361,1.070716




employment_status


,mean,count,diff,risk
employment_status,,,,
NA,0.580247,162,0.009497,1.016639
employed,0.656459,2090,0.085709,1.150170
self_employed,0.548276,580,-0.022474,0.960623
student,0.419773,617,-0.150977,0.735476
unemployed,0.435572,551,-0.135178,0.763157




location


,mean,count,diff,risk
location,,,,
NA,0.603550,169,0.032800,1.057469
africa,0.546154,390,-0.024596,0.956906
asia,0.585308,844,0.014558,1.025507
europe,0.584821,896,0.014071,1.024654
north_america,0.576923,1092,0.006173,1.010816
south_america,0.525452,609,-0.045298,0.920633


### Correlation numerical features with converted

In [18]:
df_full_train[numerical].corrwith(df_full_train.converted)

annual_income               0.188437
number_of_courses_viewed    0.393200
interaction_count           0.450273
lead_score                  0.482078
dtype: float64